# Обучение с подкреплением

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Обучение с подкреплением».

## Что делает метод

В обучении с подкреплением агент учится последовательности действий, максимизирующей итоговое вознаграждение. Он не получает готовых правильных ответов: он пробует действия, наблюдает реакцию среды и вознаграждение, и постепенно строит стратегию (политику) поведения.

Это подходит для задач управления и принятия решений: маршрут робота, расписание эксперимента, режим работы установки.

В этом блокноте мы используем простую собственную мини-среду на numpy («сеточный мир») и обучим агента методом Q-обучения — без тяжёлых сред и внешних библиотек. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы осваиваете новый прибор без инструкции. Вы нажимаете кнопки, наблюдаете за результатом («нагрев пошёл» или «ошибка»), запоминаете удачные последовательности и постепенно вырабатываете стратегию. Именно так устроено обучение с подкреплением.

Агент не получает готовых правильных ответов (в отличие от обучения с учителем). Он **действует** в среде, получает **вознаграждение** (число, показывающее, насколько хорошо он справляется), и постепенно учится действовать так, чтобы суммарное вознаграждение за весь эпизод было максимальным.

**В этом блокноте:** простой «сеточный мир» 5×5. Агент стартует в углу, цель — добраться до противоположного угла, минуя ловушки. Никаких инструкций — только опыт проб и ошибок.

**Ключевые понятия, которые встретятся в блокноте:**
- **Агент** — обучаемая система, принимающая решения (аналог экспериментатора или управляющей программы).
- **Среда** — то, с чем взаимодействует агент (лабиринт, прибор, симулятор реактора).
- **Состояние** — текущее описание ситуации (например, координаты агента на сетке).
- **Действие** — то, что агент может сделать (шаг вверх/вниз/влево/вправо).
- **Вознаграждение (reward)** — числовой сигнал после каждого действия: положительный за успех, отрицательный за ошибку или лишний шаг.
- **Политика (policy)** — стратегия: что делать в каждом состоянии. Цель обучения — найти оптимальную политику.
- **Q-таблица** — таблица оценок «ценности» каждого действия в каждом состоянии. Q(s, a) = ожидаемое суммарное вознаграждение, если в состоянии s выбрать действие a.
- **Исследование vs. использование (exploration vs. exploitation)** — дилемма: пробовать новые действия (исследование) или выбирать известное лучшее (использование). Параметр ε (epsilon) управляет этим балансом.
- **Коэффициент дисконтирования γ (gamma)** — насколько агент «дальновиден»: γ близкое к 1 означает, что он ценит отдалённые вознаграждения почти так же, как немедленные.

## 1. Установка библиотек

In [ ]:
%pip install -q numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационная среда

Сеточный мир — поле 5×5. Агент стартует в левом верхнем углу (0,0), цель — в правом нижнем (4,4). На каждом шаге доступны четыре действия: вверх, вниз, влево, вправо. Есть три клетки-ловушки (X) со штрафом −10.

**Система вознаграждений:**
- Достигнул цели: +10 (эпизод завершается)
- Попал в ловушку: −10 (эпизод завершается)
- Каждый обычный шаг: −0,2 (штраф за лишние движения — агент учится идти кратко)

**Что делает ячейка ниже:** определяет класс среды и показывает карту поля до начала обучения, чтобы вы видели расположение старта, цели и ловушек.

In [ ]:
import numpy as np

class GridWorld:
    """Простая мини-среда: поле, цель, ловушки."""
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (size - 1, size - 1)
        self.traps = {(1, 3), (2, 1), (3, 3)}
        # Действия: 0 вверх, 1 вниз, 2 влево, 3 вправо.
        self.moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    def reset(self):
        self.pos = self.start
        return self.pos

    def step(self, action):
        """Возвращает (новое состояние, вознаграждение, признак конца)."""
        dr, dc = self.moves[action]
        r = min(max(self.pos[0] + dr, 0), self.size - 1)
        c = min(max(self.pos[1] + dc, 0), self.size - 1)
        self.pos = (r, c)
        if self.pos == self.goal:
            return self.pos, 10.0, True
        if self.pos in self.traps:
            return self.pos, -10.0, True
        return self.pos, -0.2, False     # штраф за шаг побуждает идти короче


env = GridWorld()
print(f"Поле: {env.size}x{env.size}, старт {env.start}, цель {env.goal}")
print(f"Клетки-ловушки: {sorted(env.traps)}")

In [ ]:
import matplotlib.pyplot as plt

# Показываем карту сеточного мира до обучения.
fig, ax = plt.subplots(figsize=(5.2, 5.2))
ax.set_xlim(-0.5, env.size - 0.5)
ax.set_ylim(-0.5, env.size - 0.5)
ax.set_xticks(range(env.size))
ax.set_yticks(range(env.size))
ax.invert_yaxis()
ax.grid(True)
ax.set_title("Карта сеточного мира (до обучения)", fontsize=13)
ax.set_xlabel("Столбец"); ax.set_ylabel("Строка")

for r in range(env.size):
    for c in range(env.size):
        if (r, c) == env.start:
            ax.text(c, r, "старт", ha="center", va="center",
                    color=VIZ["series"][0], weight="bold", fontsize=10)
        elif (r, c) == env.goal:
            ax.text(c, r, "цель", ha="center", va="center",
                    color=VIZ["series"][1], weight="bold", fontsize=11)
        elif (r, c) in env.traps:
            ax.text(c, r, "X\nловушка", ha="center", va="center",
                    color=VIZ["series"][2], weight="bold", fontsize=9)
        else:
            ax.text(c, r, f"({r},{c})", ha="center", va="center",
                    color=VIZ["edge"], fontsize=8)

fig.tight_layout()
plt.show()

**Как читать эту карту.** «Старт» — начальная позиция агента. «Цель» — куда нужно добраться. X/ловушка — клетки, которых нужно избегать (попадание завершает эпизод штрафом). Остальные клетки — проходимые. Задача агента: найти кратчайший безопасный путь от старта до цели.

## 4. Применение метода

### Шаг 1. Инициализация Q-таблицы

**Что делает ячейка ниже:** создаёт Q-таблицу — матрицу размера (число_состояний × число_действий) = (25 × 4). Сначала все значения равны нулю: агент ничего не знает. Он заполнит таблицу в ходе обучения, постепенно уточняя оценку каждого действия в каждой клетке.

Q-таблицу можно представить как шпаргалку: «если я нахожусь в клетке (2,3) и иду вправо — ожидаемое суммарное вознаграждение составит такое-то число».

In [ ]:
n_states = env.size * env.size
n_actions = 4
Q = np.zeros((n_states, n_actions))

def state_id(pos):
    """Номер клетки как индекс в таблице Q."""
    return pos[0] * env.size + pos[1]

### Шаг 2. Обучение агента (Q-обучение)

**Что делает ячейка ниже:** запускает 400 эпизодов. В каждом эпизоде агент идёт от старта до цели (или до ловушки). На каждом шаге:
1. Выбирает действие: с вероятностью ε — случайное (исследование), иначе — лучшее по Q-таблице (использование).
2. Получает вознаграждение от среды.
3. Обновляет Q-таблицу по правилу:  
   **Q(s, a) ← Q(s, a) + α × (вознаграждение + γ × max Q(s') − Q(s, a))**  
   Это правило говорит: «подтяни оценку этого действия в сторону фактически полученного результата».

- **α (alpha)** — скорость обучения: насколько сильно каждый новый опыт меняет оценку.
- **ε (epsilon)** — убывает со временем: сначала агент много исследует, потом доверяет выученному.

Вывод: средняя награда за последние 50 эпизодов — показатель, нашёл ли агент стабильную стратегию.

In [ ]:
# Фиксируем зерно для воспроизводимости обучения (исследование на np.random)
import random
random.seed(42)
np.random.seed(42)
rng = np.random.default_rng(42)

alpha = 0.2        # скорость обучения
gamma = 0.95       # вес будущих вознаграждений
n_episodes = 400
returns = []

for episode in range(n_episodes):
    pos = env.reset()
    epsilon = max(0.05, 1.0 - episode / 250)   # доля случайных действий
    total, done, steps = 0.0, False, 0
    while not done and steps < 100:
        s = state_id(pos)
        if rng.random() < epsilon:
            a = rng.integers(n_actions)         # исследование
        else:
            a = int(np.argmax(Q[s]))            # использование
        new_pos, reward, done = env.step(a)
        s2 = state_id(new_pos)
        # Правило обновления Q-обучения.
        target = reward + gamma * np.max(Q[s2]) * (not done)
        Q[s, a] += alpha * (target - Q[s, a])
        pos = new_pos
        total += reward
        steps += 1
    returns.append(total)

print(f"Обучение завершено: {n_episodes} эпизодов.")
print(f"Средняя награда за последние 50 эпизодов: "
      f"{np.mean(returns[-50:]):.2f}")

### Шаг 3. Кривая обучения и карта выученной стратегии

**Что делает ячейка ниже:** строит два графика рядом:
1. Кривая обучения — как меняется суммарная награда за эпизод по мере обучения.
2. Карта выученной стратегии — стрелка в каждой клетке показывает лучшее действие по обученной Q-таблице.

Это ключевой «ага»-момент: мы видим, что агент нашёл путь — стрелки складываются в маршрут от старта к цели, обходящий ловушки.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Сглаженная кривая награды.
smooth = pd.Series(returns).rolling(20, min_periods=1).mean()

fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.8))

# --- Кривая обучения ---
axes[0].plot(returns, color=VIZ["series"][3], alpha=0.30,
             label="Награда за эпизод", linewidth=1.0)
axes[0].plot(smooth, color=VIZ["series"][0], label="Скользящее среднее (20)",
             linewidth=2.5)
axes[0].axhline(0, color=VIZ["edge"], linewidth=1.0, linestyle=":")
axes[0].set_title("Кривая обучения агента")
axes[0].set_xlabel("Эпизод")
axes[0].set_ylabel("Суммарная награда за эпизод")
axes[0].legend(loc="lower right")

# --- Карта стратегии + оптимальная траектория ---
symbols = {0: "↑", 1: "↓", 2: "←", 3: "→"}  # ↑ ↓ ← →
ax = axes[1]
ax.set_xlim(-0.5, env.size - 0.5)
ax.set_ylim(-0.5, env.size - 0.5)
ax.set_xticks(range(env.size))
ax.set_yticks(range(env.size))
ax.invert_yaxis()
ax.grid(True)
ax.set_title("Выученная стратегия и оптимальный маршрут")
ax.set_xlabel("Столбец")
ax.set_ylabel("Строка")

for r in range(env.size):
    for c in range(env.size):
        if (r, c) == env.goal:
            ax.text(c, r, "цель", ha="center", va="center",
                    color=VIZ["series"][1], weight="bold", fontsize=11)
        elif (r, c) in env.traps:
            ax.text(c, r, "X", ha="center", va="center",
                    color=VIZ["series"][2], weight="bold", fontsize=16)
        else:
            a = int(np.argmax(Q[state_id((r, c))]))
            ax.text(c, r, symbols[a], ha="center", va="center",
                    color=VIZ["series"][0], fontsize=18)

# Проследим оптимальный путь агента.
pos = env.reset()
path = [pos]
done = False
steps = 0
while not done and steps < 30:
    a = int(np.argmax(Q[state_id(pos)]))
    pos, _, done = env.step(a)
    path.append(pos)
    steps += 1
# Рисуем маршрут.
if len(path) > 1:
    xs = [p[1] for p in path]
    ys = [p[0] for p in path]
    ax.plot(xs, ys, color=VIZ["series"][2], linewidth=2.5,
            marker="o", markersize=5, zorder=5, alpha=0.7,
            label="Оптимальный маршрут")
    ax.legend(loc="upper right", fontsize=9)

fig.tight_layout()
plt.show()

print(f"Длина оптимального маршрута: {len(path) - 1} шагов")
print(f"Средняя награда за последние 50 эпизодов: {np.mean(returns[-50:]):.2f}")

**Как читать эти графики.**

- **Кривая обучения (слева):** ось X — номер эпизода, ось Y — суммарное вознаграждение за эпизод. Серые точки — отдельные эпизоды (шумные). Синяя кривая — скользящее среднее за 20 эпизодов. В начале награда отрицательная или близкая к нулю (агент часто попадает в ловушки или блуждает). По мере обучения скользящее среднее растёт и стабилизируется — агент нашёл хорошую стратегию. Горизонтальная пунктирная линия на 0 — ориентир.

- **Карта стратегии (справа):** стрелка в каждой клетке — то, что агент считает лучшим действием. «Хорошая» стратегия: стрелки складываются в маршрут от «старт» до «цель», не проходящий через X. Оранжевая линия — фактический маршрут агента при жадном следовании выученной стратегии. Если маршрут доходит до цели, не попадая в ловушку — обучение прошло успешно.

## 5. Интерпретация результата

- **Кривая обучения**: суммарная награда за эпизод растёт и выходит на плато. Рост означает, что агент нашёл стратегию лучше случайной; плато — что стратегия стабилизировалась.
- **Карта стратегии**: стрелка в клетке — действие, которое агент считает лучшим. Стрелки должны складываться в маршрут от старта к цели, обходящий клетки-ловушки (X).
- **Параметр `epsilon`** управляет балансом исследования и использования: в начале агент много пробует случайно, со временем всё больше доверяет выученной таблице. Без исследования агент рискует застрять в неоптимальной стратегии.
- **`gamma`** задаёт дальновидность: близкое к единице значение заставляет учитывать отдалённые последствия действий.

Помните: метод обучается на собственном опыте взаимодействия со средой. Качество стратегии ограничено тем, насколько разнообразные ситуации агент успел опробовать.

## 5а. Поэкспериментируйте сами

| Что изменить | Где | Ожидаемый эффект |
|---|---|---|
| `n_episodes = 100` вместо 400 | ячейка обучения | Агент не успевает обучиться: стратегия слабее, маршрут хуже |
| `gamma = 0.5` вместо 0.95 | ячейка обучения | Агент «близоруко» смотрит только на ближайшие шаги; стратегия может быть субоптимальной |
| Добавить ловушку: `self.traps = {(1,3),(2,1),(3,3),(0,2)}` | класс GridWorld | Больше препятствий — нужно больше эпизодов или другой маршрут |
| `alpha = 0.5` вместо 0.2 | ячейка обучения | Быстрее обучается, но менее стабильно (больше шума в кривой) |

## 6. Попробуйте на своих данных

Адаптируйте мини-среду под свою задачу принятия решений.

1. Опишите свою среду по образцу класса `GridWorld`: методы `reset` и `step`, набор состояний и действий, правило вознаграждения.
2. Награждение — ключевой элемент: оно должно поощрять желаемый исход и штрафовать нежелательный.
3. Подберите число эпизодов, `alpha`, `gamma` и расписание `epsilon`.
4. Выполните ячейки раздела 4.

In [ ]:
# --- Шаблон собственной среды ---
# class MyEnv:
#     def reset(self):
#         self.state = ...            # начальное состояние
#         return self.state
#     def step(self, action):
#         # примените действие, вычислите новое состояние
#         reward = ...                # вознаграждение за шаг
#         done = ...                  # достигнут ли конец эпизода
#         return self.state, reward, done
#
# Затем повторите цикл обучения раздела 4 с объектом MyEnv.

## 7. Краткие выводы

- Обучение с подкреплением — это обучение через пробы и ошибки: агент не получает правильных ответов, а сам нащупывает удачную стратегию.
- Q-обучение хранит оценку «ценности» каждого действия в каждом состоянии и постепенно уточняет её по полученным вознаграждениям.
- Баланс исследования и использования (параметр ε) критичен: без исследования агент застревает в первом найденном решении.
- Дизайн функции вознаграждения — ключевой этап: неправильно заданная награда приводит к нежелательному поведению.
- Для реальных научных задач (управление установкой, параметры синтеза) нужен быстрый симулятор среды — прямое обучение на реальном оборудовании требует слишком много проб.
- Табличное Q-обучение работает только для небольших дискретных пространств; для сложных задач используют глубокое RL (DQN, PPO) из библиотек Stable Baselines3 / CleanRL.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. Кривая обучения вышла на плато с суммарной наградой около −2, однако на карте стратегии часть стрелок всё ещё ведёт в ловушку. Что могло помешать агенту найти оптимальный маршрут и какие два параметра вы скорректируете?
2. Вы применяете Q-обучение к задаче управления лабораторной установкой, у которой 50 непрерывных параметров состояния. Почему табличное Q-обучение непригодно в этом случае и какой класс методов следует рассмотреть?
3. Агент обучается управлять химическим реактором и получает вознаграждение +10 только при достижении целевого выхода продукта. Во всех остальных шагах вознаграждение равно нулю. Какую проблему порождает такое разреженное вознаграждение и как её смягчить, не меняя целевого критерия?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Наиболее вероятные причины: ε снизился слишком быстро, и агент преждевременно «закрепил» субоптимальную политику (недостаточное исследование), либо скорость обучения α слишком мала, чтобы Q-значения успели распространиться от целевых состояний к начальным за 400 эпизодов. Стоит увеличить `n_episodes` или замедлить убывание ε.
2. Табличное Q-обучение требует хранить Q-значение для каждой пары (состояние, действие), что при непрерывном пространстве состояний физически невозможно. Необходимо использовать методы глубокого обучения с подкреплением (DQN, PPO, SAC), где Q-функция или политика аппроксимируются нейронной сетью.
3. Разреженное вознаграждение делает обучение крайне медленным: агент случайно блуждает, почти никогда не достигая цели, и не получает обучающего сигнала. Смягчение без изменения целевого критерия: добавить небольшое промежуточное вознаграждение («reward shaping») за движение в сторону цели по измеримым промежуточным признакам (например, рост концентрации промежуточного продукта), используя разность потенциалов, чтобы не изменить оптимальную политику.
</details>